# 03 · Label Validation and Consolidation

**The scientific core. Reads `labels_raw.csv` from notebook 02.**

Four independent checks, each able to fail on its own. Then consolidation to one
label per photograph for everything downstream.

| Check | Question | Paper result |
|---|---|---|
| 1 · Expert agreement | Do clinicians agree with the derived grade? | kappa 0.1534 (slight) |
| 2 · Negative control | Does healthy skin get graded severe? | 86.3% yes |
| 3 · Stability | Do a photograph's own copies agree? | 19.1% disagree |
| 4 · Plausibility | Is the class mix clinically real? | 75% severe |

## Two checks need external files
Expert agreement needs a CSV of clinician grades. The negative control needs the
Kaggle healthy-skin folder. If either is missing, that check is **flagged as
unverifiable** and its paper value is recorded, rather than skipped silently.
Set `EXPERT_CSV` and `KAGGLE_HEALTHY` in Cell 1 to reproduce them.

## Consolidation
Once instability is measured, each photograph's copies are collapsed to one label
by applying the threshold rule to the mean of their proportions. This is what
makes the label set consistent for notebooks 04 onward, and it is done *after*
the stability check, not before, so the check still sees the copies.

## Outputs
- `labels_consolidated.csv` — one row per photograph
- `validation_report.json` — every number, with each check marked confirmed or flagged


In [ ]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json

INTERIM = Path('data/interim')
LABELS  = INTERIM / 'labels_raw.csv'

# ── optional inputs for two of the four checks ───────────────────────
# Expert agreement needs a CSV of clinician grades. Negative control needs
# the Kaggle healthy-skin folder. If either is absent, that check is
# FLAGGED as unverifiable rather than skipped silently.
EXPERT_CSV        = INTERIM / 'expert_review.csv'        # cols: filename, expert_grade
KAGGLE_HEALTHY    = Path('data/raw/kaggle_dfu/Normal')   # healthy patches
# ─────────────────────────────────────────────────────────────────────

RESIZE, N_INIT, SEED = 128, 4, 42

if not LABELS.exists():
    print(f'STOPPING. {LABELS} not found. Run 02_label_derivation.ipynb first.')
    raise SystemExit(1)

lab = pd.read_csv(LABELS)
lab = lab[lab.severity != 'error'].copy()
report = {}   # accumulates every number, saved to JSON at the end
print(f'loaded {len(lab):,} labelled images, '
      f'{lab.photo_unit.nunique():,} photographs')

In [ ]:
# Cell 2 · CHECK 1 — expert agreement
# 150 images were spot-checked by two clinicians. Filenames have a leading
# NNN_ index prefix that must be stripped before matching. 16 of the 150
# are from the Kaggle corpus (simple names like 105.jpg) and won't appear
# in labels_raw, which is Roboflow-only. Kappa is computed on the matchable
# Roboflow subset (~134 images).
import re
from sklearn.metrics import cohen_kappa_score

gmap = {'mild': 0, 'moderate': 1, 'severe': 2}
COMPILED = INTERIM / 'expert_review_compiled.csv'
SINGLE   = INTERIM / 'expert_review.csv'

def strip_prefix(fn):
    # remove leading NNN_ spot-check index, e.g. "001_" from "001_filename.jpg"
    return re.sub(r'^\d+_', '', Path(str(fn)).name)

def kap(a, b, w=None):
    m = a.notna() & b.notna()
    return cohen_kappa_score(a[m], b[m], weights=w) if m.sum() else float('nan')

lab['_fname'] = lab.path.map(lambda p: Path(p).name)

if COMPILED.exists():
    exp = pd.read_csv(COMPILED)
    exp['_fname'] = exp.filename.map(strip_prefix)
    # separate Roboflow (matchable) from Kaggle (not in labels_raw)
    is_rf = exp._fname.str.contains('_jpg.rf.', na=False)
    n_kaggle = int((~is_rf).sum())
    m = lab.merge(exp[is_rf], on='_fname', how='inner')
    print(f'CHECK 1 — expert agreement')
    print(f'  total spot-check images : 150')
    print(f'  Kaggle (not matchable)  : {n_kaggle}  '
          f'(simple names, not in labels_raw)')
    print(f'  Roboflow available      : {int(is_rf.sum())}')
    print(f'  actually matched        : {len(m)}')
    if len(m):
        derived = m.severity.map(gmap)
        A = m.expert_a_grade.str.lower().map(gmap)
        B = m.expert_b_grade.str.lower().map(gmap)
        kA  = kap(derived, A)
        kB  = kap(derived, B)
        kAB = kap(A, B)
        print(f'\n  derived vs Expert A : {kA:+.4f}')
        print(f'  derived vs Expert B : {kB:+.4f}')
        print(f'  Expert A vs B       : {kAB:+.4f}   <- inter-rater')
        print(f'\n  Note: the paper reports 0.1534, which is a decision-tree')
        print(f'  recalibration fitted to expert labels — a separate procedure.')
        print(f'  The raw held-out agreement above is what this notebook produces.')
        report['check1_expert'] = dict(
            status='confirmed', n_matched=int(len(m)),
            n_kaggle_excluded=n_kaggle,
            kappa_derived_vs_A=float(kA),
            kappa_derived_vs_B=float(kB),
            kappa_inter_rater=float(kAB),
            paper_recalibrated_kappa=0.1534)
    else:
        print('  FLAGGED: Roboflow names present but none matched labels_raw.')
        print('  Check that labels_raw.csv paths use the original Roboflow filenames.')
        report['check1_expert'] = dict(status='flagged',
                                       reason='no filename overlap after prefix strip')
elif SINGLE.exists():
    exp = pd.read_csv(SINGLE)
    exp['_fname'] = exp.filename.map(strip_prefix)
    m = lab.merge(exp, on='_fname', how='inner')
    print(f'CHECK 1 — expert agreement (single expert, matched {len(m)})')
    if len(m):
        k = kap(m.severity.map(gmap), m.expert_grade.str.lower().map(gmap))
        print(f'  derived vs expert : {k:+.4f}')
        report['check1_expert'] = dict(status='confirmed', n=int(len(m)), kappa=float(k))
    else:
        print('  FLAGGED: no match after prefix strip.')
        report['check1_expert'] = dict(status='flagged', reason='no overlap')
else:
    print('CHECK 1 — FLAGGED as unverifiable.')
    print(f'  Neither expert CSV found in {INTERIM}')
    report['check1_expert'] = dict(status='flagged',
                                   reason='expert CSV not provided',
                                   paper_recalibrated_kappa=0.1534,
                                   paper_inter_rater=0.27)

In [ ]:
# Cell 3 · CHECK 2 — negative control on healthy skin
# Run the identical derivation on images that contain no ulcer. The correct
# necrotic fraction is zero. Anything graded severe is a false positive the
# method cannot see on real wounds, where there is no ground truth to check.
from PIL import Image
from skimage import color
from sklearn.cluster import KMeans
from joblib import Parallel, delayed
import os

def tissue_proportions(path):
    # identical to notebook 02, repeated so this notebook stands alone
    try:
        with Image.open(path) as im:
            a = np.asarray(im.convert('RGB').resize((RESIZE, RESIZE)),
                           dtype=np.float32) / 255.0
    except Exception:
        return np.array([np.nan, np.nan, np.nan])
    l = color.rgb2lab(a).reshape(-1, 3)
    km = KMeans(3, n_init=N_INIT, random_state=SEED).fit(l)
    o = np.argsort(km.cluster_centers_[:, 0])
    c = np.bincount(km.labels_, minlength=3)[o].astype(np.float64)
    return c / c.sum()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
if KAGGLE_HEALTHY.exists():
    healthy = [p for p in KAGGLE_HEALTHY.rglob('*') if p.suffix.lower() in IMG_EXT]
    print(f'CHECK 2 — negative control')
    print(f'  healthy skin images: {len(healthy):,}')
    props = np.vstack(Parallel(n_jobs=max(1,(os.cpu_count() or 2)-1))(
        delayed(tissue_proportions)(p) for p in healthy))
    nec = props[:, 0]
    nec = nec[~np.isnan(nec)]
    severe = (nec >= 0.20).mean() * 100
    print(f'  graded SEVERE      : {severe:.1f}%   (correct answer: 0%)')
    print(f'  mean necrosis      : {nec.mean():.3f}')
    print(f'  max necrosis       : {nec.max():.3f}   (on intact skin)')
    report['check2_negctrl'] = dict(status='confirmed', n=len(healthy),
                                    pct_severe=float(severe),
                                    mean_necrosis=float(nec.mean()),
                                    max_necrosis=float(nec.max()))
else:
    print('CHECK 2 — FLAGGED as unverifiable.')
    print(f'  {KAGGLE_HEALTHY} not found.')
    print('  The paper reports 86.3% of healthy skin graded severe.')
    print('  Point KAGGLE_HEALTHY at the Kaggle Normal folder to reproduce it.')
    report['check2_negctrl'] = dict(status='flagged',
                                    reason='healthy folder not provided',
                                    paper_value_pct_severe=86.3)

In [ ]:
# Cell 4 · CHECK 3 — stability under augmentation
# A photograph and its augmented copies depict the same wound, so they must
# receive the same grade. Group by content cluster and count disagreements.
grp = lab.groupby('photo_unit')
multi = grp.filter(lambda g: len(g) > 1)
n_multi = multi.photo_unit.nunique()

def grade_range(g):
    return g.severity.nunique()

inconsistent = grp.filter(lambda g: len(g) > 1 and g.severity.nunique() > 1)
n_incon = inconsistent.photo_unit.nunique()
pct_incon = n_incon / max(n_multi, 1) * 100

print('CHECK 3 — stability under augmentation')
print(f'  photographs with >1 copy      : {n_multi:,}')
print(f'  carrying more than one grade  : {n_incon:,} ({pct_incon:.1f}%)')

# the mechanism: instability concentrates at the threshold. Bin each
# photograph by how far its mean necrosis sits from the 0.20 cut.
ph = grp.agg(nec_mean=('necrosis_prop', 'mean'),
             n_grades=('severity', 'nunique'), n=('severity', 'size'))
ph = ph[ph.n > 1]
ph['dist'] = (ph.nec_mean - 0.20).abs()
print('\n  flip rate by distance from the 0.20 threshold:')
for lo, hi in [(0, 0.02), (0.02, 0.05), (0.05, 0.10), (0.10, 0.20), (0.20, 1)]:
    m = (ph.dist >= lo) & (ph.dist < hi)
    if m.sum():
        flip = (ph.loc[m, 'n_grades'] > 1).mean() * 100
        print(f'    {lo:.2f}-{hi:.2f} : {flip:5.1f}%  (n={m.sum():,})')
report['check3_stability'] = dict(status='confirmed', n_multi_copy=int(n_multi),
                                  n_inconsistent=int(n_incon),
                                  pct_inconsistent=float(pct_incon))

In [ ]:
# Cell 5 · CHECK 4 — corpus plausibility
# If the derivation were valid, the class distribution would resemble a real
# DFU cohort. It does not.
mean_nec = lab.necrosis_prop.mean()
pct_above = (lab.necrosis_prop >= 0.20).mean() * 100
dist = lab.severity.value_counts(normalize=True) * 100

print('CHECK 4 — plausibility')
print(f'  mean necrotic fraction  : {mean_nec:.3f}')
print(f'  images above severe cut : {pct_above:.1f}%')
print('  class distribution:')
for c in ['mild', 'moderate', 'severe']:
    print(f'    {c:<9} {dist.get(c,0):5.1f}%')
print('\n  No clinical DFU cohort is three-quarters severe. Combined with the')
print('  negative control, this points to the derivation manufacturing')
print('  necrosis from shadow, depth and skin tone rather than measuring it.')
report['check4_plausibility'] = dict(status='confirmed',
                                     mean_necrosis=float(mean_nec),
                                     pct_above_threshold=float(pct_above),
                                     dist_severe_pct=float(dist.get('severe', 0)))

In [ ]:
# Cell 6 · consolidate to one label per photograph
# Now that instability has been measured, collapse each photograph's copies
# to a single label by applying the threshold rule to the MEAN proportions.
# This removes the inconsistency for everything downstream.
cons = lab.groupby('photo_unit').agg(
    photo_id=('photo_id', 'first'),
    hash_cluster=('hash_cluster', 'first'),
    patient_id=('patient_id', 'first'),
    n_copies=('path', 'size'),
    necrosis_prop=('necrosis_prop', 'mean'),
    slough_prop=('slough_prop', 'mean'),
    granulation_prop=('granulation_prop', 'mean'),
    representative=('path', 'first')).reset_index()

def grade(r):
    if r.necrosis_prop >= 0.20: return 'severe'
    if r.granulation_prop >= 0.67 and r.slough_prop <= 0.27: return 'mild'
    return 'moderate'

cons['severity'] = cons.apply(grade, axis=1)
cons['y'] = cons.severity.map({'mild': 0, 'moderate': 1, 'severe': 2})

# confirm consolidation actually removed the inconsistency
before = report['check3_stability']['n_inconsistent']
print(f'consolidated {len(lab):,} images -> {len(cons):,} photographs')
print(f'  inconsistent photographs before : {before:,}')
print(f'  after consolidation             : 0 (one label per photograph now)')
print('\n  consolidated class distribution:')
print('  ' + str(cons.severity.value_counts().to_dict()))
report['consolidation'] = dict(n_images=len(lab), n_photographs=len(cons),
                               dist=cons.severity.value_counts().to_dict())

In [ ]:
# Cell 7 · save outputs
cons.to_csv(INTERIM / 'labels_consolidated.csv', index=False)
print(f'wrote {(INTERIM / "labels_consolidated.csv").resolve()}')
print(f'  {len(cons):,} photographs, one label each')

with open(INTERIM / 'validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'wrote {(INTERIM / "validation_report.json").resolve()}')

In [ ]:
# Cell 8 · verdict
print('=' * 60)
print('STAGE 03 COMPLETE — four validation checks')
print('=' * 60)
labels = {'check1_expert': 'expert agreement',
          'check2_negctrl': 'negative control',
          'check3_stability': 'stability',
          'check4_plausibility': 'plausibility'}
for k, name in labels.items():
    st = report[k]['status']
    mark = 'CONFIRMED' if st == 'confirmed' else 'FLAGGED  '
    print(f'  [{mark}] {name}')
n_flag = sum(report[k]['status'] == 'flagged' for k in labels)
print(f'\n  {4-n_flag}/4 checks reproduced here.')
if n_flag:
    print(f'  {n_flag} flagged for missing external files — list these in the')
    print('  verification log; a flagged gap is honest, not a failure.')
print('\n  consolidated labels are one per photograph, downstream-clean.')
print('\nnext: 04_split.ipynb')